# Employee Sentiment Analysis

A data-driven analysis of an authorized employee message CSV. This notebook imports and orchestrates the project's existing modules in `src/`; analysis and file generation live in reusable Python functions. No data or results are embedded in this notebook. Keep `outputs/` private because the labeled-message export contains message text.

## 1. Setup and project imports

Install `requirements.txt`, start Jupyter from the project root, and run each cell in order. The raw dataset must be a real `.csv` named `test.csv`. The known `test.csv.html` viewer page is deliberately not accepted as input.

In [ ]:
from pathlib import Path
import json
import logging
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir() and (ROOT.parent / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

# Explicitly connect every project module.
from src.sentiment import inspect_dataframe, resolve_column, classify_messages
from src.scoring import monthly_scores, monthly_rankings
from src.flight_risk import find_flight_risks
from src.modeling import make_forecast_frame, evaluate_forecast
from src.analysis import run_analysis, MODEL_NAME

DATA_PATH = ROOT / 'test.csv'
OUTPUT_DIR = ROOT / 'outputs'
VISUALIZATIONS_DIR = ROOT / 'visualizations'
print('Project root:', ROOT)
print('CSV available:', DATA_PATH.is_file())

## 2. Load and inspect the actual schema

This cell reads only `test.csv`. It prints actual headers, types, missing counts, duplicate count, and field candidates. The notebook uses a candidate automatically only when exactly one header matches the relevant hints; otherwise set the exact observed column name in the configuration variables. No column names are assumed.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Actual CSV not found at {DATA_PATH}. Add the authorized test.csv; test.csv.html is not a CSV input.')
raw = pd.read_csv(DATA_PATH, low_memory=False)
profile = inspect_dataframe(raw)
print(json.dumps(profile, indent=2, default=str))

# After reviewing the real headers above, set any ambiguous field explicitly.
TEXT_COL = None
EMPLOYEE_COL = None
DATE_COL = None
TEXT_COL = resolve_column(raw, TEXT_COL, 'text')
EMPLOYEE_COL = resolve_column(raw, EMPLOYEE_COL, 'employee')
DATE_COL = resolve_column(raw, DATE_COL, 'date')
print({'text_column': TEXT_COL, 'employee_column': EMPLOYEE_COL, 'date_column': DATE_COL})

## 3. Sentiment analysis

Uses Hugging Face Transformers with `cardiffnlp/twitter-roberta-base-sentiment-latest`. Predictions map to Positive, Neutral, or Negative; blank text stays unlabeled. Labels are weak model predictions, and the model's social-media training may not reflect workplace email context. The pipeline exports a pseudonymous employee ID and retains message text only in private `outputs/labeled_messages.csv`.

## 4. EDA, data validation, and visualizations

The integrated pipeline reports invalid dates, missing identities, blank text, duplicate rows, sentiment distribution, monthly activity, monthly sentiment trends, and activity outliers. All charts are saved under `visualizations/`; all CSV outputs are written under `outputs/`.

## 5. Monthly employee sentiment scores and rankings

Positive = +1, Neutral = 0, Negative = -1. Scores are summed by pseudonymous employee and calendar month, with message-level cumulative values resetting each month. Each month has separate top-positive and top-negative ranking files; ties sort alphabetically by employee ID.

## 6. Rolling 30-day flight risk

A qualifying interval has at least four negative messages whose first and fourth timestamps are no more than 30 elapsed days apart. Calendar-month boundaries do not reset this calculation. The report retains overlapping qualifying four-message intervals so each reported interval itself satisfies the rule.

## 7. Predictive modeling

Linear Regression predicts an employee's score in their next observed active month. Features use earlier activity, word count, message length, and sentiment history. Evaluation uses a chronological month holdout and reports MAE, MSE, RMSE, and R? (when the test set has at least two records). Sparse histories are reported as insufficient rather than fabricated.

## 8. Run the integrated pipeline and review its validations

`run_analysis` connects all modules, validates monthly score reconciliation, ranking limits, and flight-risk interval constraints, then writes the report and artifacts.

In [ ]:
results = run_analysis(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    visualizations_dir=VISUALIZATIONS_DIR,
    text_col=TEXT_COL,
    employee_col=EMPLOYEE_COL,
    date_col=DATE_COL,
    model_name=MODEL_NAME,
)
print('Detected columns:', results['columns'])
print('Sentiment counts:', results['sentiment_counts'])
print('Validation checks:', results['validation'])
print('Forecast metrics:', json.dumps(results['forecast_metrics'], indent=2, default=str))
display(results['monthly_scores'].head(10))
display(results['positive_rankings'].head(10))
display(results['negative_rankings'].head(10))
display(results['flight_risks'].head(20))
display(results['activity_anomalies'].head(20))

## 9. Review generated artifacts and interpretation limits

Summary files use pseudonymous IDs. The labeled dataset includes message text and must remain access-controlled. Sentiment scores are not a measure of performance or wellbeing; a flight-risk interval is a review signal only. The forecast is exploratory, not causal.

In [ ]:
print('Outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print('-', path.name)
print('Visualizations:')
for path in sorted(VISUALIZATIONS_DIR.glob('*.png')):
    print('-', path.name)